# Part 1 measured results and review aggregation

This notebook computes every exported result from immutable execution records.
The question set is an AI-assisted draft; the revised candidate is a development-set evaluation after recorded fixes. AI source-content judgments are provisional and are **not** either member's independent groundedness scores. Missing ratings remain null. This Windows RTX4090 run does not establish a physical T4/P100 run, Kaggle submission or hidden-set score.

Set `DATA266_PART1_RUN_DIR` to a complete run folder. After both students finish their independent forms, import their JSON with `scripts/p1_review_forms.py import`, set `DATA266_PART1_HUMAN_REVIEW` to the merged review path, and rerun this notebook. No inference or model loading occurs here.


In [1]:
import os, sys, json, hashlib, platform
from pathlib import Path
from collections import Counter
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/p1_evaluation.py').is_file())
sys.path.insert(0, str(ROOT))
from src.p1_evaluation import load_specification, summarize, digest
bundled_run = ROOT/'eval/part1_windows_run'
local_pointer = json.loads((ROOT/'eval/part1_run_pointer.json').read_text(encoding='utf-8'))
default_run = bundled_run if bundled_run.is_dir() else ROOT/local_pointer['run_dir']
RUN = Path(os.environ.get('DATA266_PART1_RUN_DIR', default_run))
OUTPUT = Path(os.environ.get('DATA266_PART1_METRICS_OUTPUT', RUN/'measured_report'))
OUTPUT.mkdir(parents=True, exist_ok=True)
spec = load_specification(ROOT/'eval/p1_golden_v2.json')
manifest = json.loads((RUN/'execution_manifest.json').read_text(encoding='utf-8'))
assert manifest['status'] == 'completed'
print('Python:', platform.python_version())
print('Inference commit:', manifest['source_commit'])
print('Inference notebook hash:', manifest['source_notebook_sha256'])
print('Questions canonical hash:', digest(spec))
print('Host:', manifest['gpu'])
print('Interpretation: recorded development-set evaluation; no independent held-out or physical T4/P100 claim.')


Python: 3.11.9
Inference commit: 55942bef55fcf15057911f6b33e93eae028d28a3
Inference notebook hash: ec8068e12b6b622f1370ea218a536e009b0a57e939aab50edaaea333ee1c2818
Questions canonical hash: e4cf051593f3665cec1e411b5b745c64e33b92e9369d25c508f71e6fa01e1763
Host: NVIDIA GeForce RTX 4090, 24564 MiB, 610.60
Interpretation: recorded development-set evaluation; no independent held-out or physical T4/P100 claim.


In [2]:
runs, reviews, rows = {}, {}, []
for mode in ('dense_only', 'final'):
    runs[mode] = json.loads((RUN/'evaluation'/f'{mode}.json').read_text(encoding='utf-8'))
    review_path = RUN/'evaluation'/f'{mode}.review.ai.json'
    if not review_path.exists():
        review_path = RUN/'evaluation'/f'{mode}.review.json'
    if mode == 'final' and os.environ.get('DATA266_PART1_HUMAN_REVIEW'):
        review_path = Path(os.environ['DATA266_PART1_HUMAN_REVIEW'])
    reviews[mode] = json.loads(review_path.read_text(encoding='utf-8'))
    rows.append(summarize(runs[mode], spec, reviews[mode]))
    print(mode, 'content review:', reviews[mode].get('content_review_method'))
    print(mode, 'human reviewers:', reviews[mode].get('reviewer_a_name'), reviews[mode].get('reviewer_b_name'))
results = pd.DataFrame(rows).set_index('mode')
display(results.T)
results.to_csv(OUTPUT/'p1_results_table.csv')
(OUTPUT/'p1_results_table.json').write_text(json.dumps(rows,indent=2),encoding='utf-8')
assert all(row['denominator'] == 25 for row in rows)
print('Unfilled human scores remain missing; AI judgments are provisional.')


dense_only content review: AI-assisted manual comparison with original course PDF text and rendered charts. Correctness checks complete requested content; each cited page is checked for its attributed material claims, with unsupported computed comparisons failing support. Provisional pending team confirmation; not either member human groundedness rating.
dense_only human reviewers: None None
final content review: AI-assisted manual comparison with original course PDF text and rendered charts. Correctness checks complete requested content; each cited page is checked for its attributed material claims, with unsupported computed comparisons failing support. Provisional pending team confirmation; not either member human groundedness rating.
final human reviewers: None None


mode,dense_only,final
denominator,25,25
answer_correct_out_of_25,6,18
answer_correctness,0.24,0.72
retrieval_recall_at_5,0.472222,0.763889
retrieval_denominator,24,24
citation_accuracy,0.140625,0.489796
citation_denominator,64,49
groundedness_mean,None,None
groundedness_agree_within_one,None,None
median_seconds_per_question,6.108288,0.039253


Unfilled human scores remain missing; AI judgments are provisional.


In [3]:
audit = json.loads((RUN/'p1_run_audit.json').read_text(encoding='utf-8'))
gpu = json.loads((RUN/'gpu_memory.json').read_text(encoding='utf-8'))
practice = pd.read_csv(RUN/'submission.csv',dtype=str,keep_default_na=False)
code_nb = json.loads((RUN/'p1_rag.executed.ipynb').read_text(encoding='utf-8'))
code_cells = [c for c in code_nb['cells'] if c['cell_type']=='code']
assert all(c.get('execution_count') is not None for c in code_cells)
assert not [o for c in code_cells for o in c.get('outputs',[]) if o.get('output_type')=='error']
safety_rows=[]
for mode, run in runs.items():
    for record in run['records']:
        a=record['turn']['answer']
        safety_rows.append(dict(mode=mode,question_id=record['question']['question_id'],
            appendix_page_cited='DOC3:p7' in a['citations'],
            appendix_figure_cited='Figure C-1' in a['figure_references'],
            answer_status=a['status'],word_count=len(a['answer'].split())))
safety=pd.DataFrame(safety_rows)
assert not safety['appendix_page_cited'].any()
assert not safety['appendix_figure_cited'].any()
assert safety['word_count'].max() <= 120
assert gpu['peak_reserved_gib'] <= gpu['allocator_budget_gib']
observed=dict(practice_rows=len(practice),practice_conversations=practice['conversation_id'].nunique(),
    practice_status_counts=dict(Counter(practice['status'])),
    inference_code_cells=len(code_cells),all_inference_cells_executed=True,
    golden_executions=len(safety),appendix_citations=int(safety['appendix_page_cited'].sum()),
    appendix_figure_references=int(safety['appendix_figure_cited'].sum()),
    maximum_golden_answer_words=int(safety['word_count'].max()),gpu=gpu,
    elapsed_notebook_seconds=manifest['elapsed_seconds'],audit=audit)
print(json.dumps(observed,indent=2))
(OUTPUT/'observed_execution.json').write_text(json.dumps(observed,indent=2),encoding='utf-8')
safety.to_csv(OUTPUT/'per_question_safety.csv',index=False)


{
  "practice_rows": 40,
  "practice_conversations": 35,
  "practice_status_counts": {
    "answered": 27,
    "not_found": 6,
    "conflict": 2,
    "false_premise": 1,
    "session_override": 4
  },
  "inference_code_cells": 46,
  "all_inference_cells_executed": true,
  "golden_executions": 50,
  "appendix_citations": 0,
  "appendix_figure_references": 0,
  "maximum_golden_answer_words": 97,
  "gpu": {
    "peak_allocated_gib": 13.157339096069336,
    "peak_reserved_gib": 13.8359375,
    "allocator_budget_gib": 14.0,
    "hardware": "NVIDIA GeForce RTX 4090",
    "actual_t4_p100_verified": false
  },
  "elapsed_notebook_seconds": 428.26600000000326,
  "audit": {
    "row_count": 40,
    "conversation_count": 35,
    "status_distribution": {
      "answered": 27,
      "not_found": 6,
      "conflict": 2,
      "false_premise": 1,
      "session_override": 4
    },
    "rows_with_citations": 32,
    "citation_presence_rate": 0.8,
    "rows_with_figures": 29,
    "median_seconds_per_qu

In [4]:
def cell(value):
    if value is None: return 'Pending'
    if isinstance(value,float): return f'{value:.4f}'
    return str(value)
fields=[('Correct / 25 (provisional AI content review)','answer_correct_out_of_25'),
        ('Correct fraction','answer_correctness'),('Recall@5','retrieval_recall_at_5'),
        ('Evidence-bearing retrieval cases','retrieval_denominator'),
        ('Citation accuracy (provisional AI source review)','citation_accuracy'),
        ('Emitted citations reviewed','citation_denominator'),
        ('Median seconds / question','median_seconds_per_question'),
        ('Two-member mean groundedness','groundedness_mean'),
        ('Two-member agreement within one point','groundedness_agree_within_one')]
table='| Measurement | Dense only | Final |\n| --- | ---: | ---: |\n'
for label,key in fields: table+=f"| {label} | {cell(rows[0][key])} | {cell(rows[1][key])} |\n"
report=f"""# Part 1 measured results: review draft

Inference commit: `{manifest['source_commit']}`. Question set: `{spec['set_id']}`.
These are measured development-set results on Windows RTX4090. Content/citation review is explicitly AI-assisted and provisional; it is not the required two-member human evaluation. The question draft needs team authorship, originality and reference review. The retained initial run and recorded fixes prevent a claim that this is an independent holdout.

{table}

The practice output has {observed['practice_rows']} rows in {observed['practice_conversations']} conversations and passed the official provided format validator. Format validation does not establish answer correctness. The complete inference notebook has {observed['inference_code_cells']} executed code cells, with no error outputs. Its wall time was {manifest['elapsed_seconds']:.2f} seconds, including setup, practice and both evaluation modes.

Peak PyTorch allocated memory was {gpu['peak_allocated_gib']:.3f} GiB and peak reserved memory was {gpu['peak_reserved_gib']:.3f} GiB under a {gpu['allocator_budget_gib']:.1f} GiB allocator budget. Text/visual retrieval ran on CPU and Qwen ran on CUDA device 0. This capacity-limited run does not verify actual T4/P100 hardware, runtime or Kaggle submission.

Across the {observed['golden_executions']} golden executions, emitted DOC3:p7 citations and Figure C-1 references were both {observed['appendix_citations']}; the maximum answer length was {observed['maximum_golden_answer_words']} whitespace-separated words. Appendix exclusion is prevention: its hidden image instruction was not supplied as trusted model input. No claim of robustness to arbitrary unseen image injections follows.

Recall@5 uses the frozen interleaved page/visual ranking before structured evidence expansion; its denominator excludes the no-evidence question. Citation judgments inspect material claim support, not mere source presence. Review notes and actual outputs are retained for team confirmation. Human metrics remain pending until two distinct team members submit all ratings.

All numerical values in this document are generated by this notebook from saved records. Do not type replacement results into the report. The team still needs to interpret observed failures in its own words, complete independent ratings, review the AI-assisted question set, and verify the tagged Kaggle run and final selected submission.
"""
(OUTPUT/'PART1_MEASURED_RESULTS.md').write_text(report,encoding='utf-8')
from IPython.display import Markdown, display
display(Markdown(report))
print('Measured artifacts:', OUTPUT)


# Part 1 measured results: review draft

Inference commit: `55942bef55fcf15057911f6b33e93eae028d28a3`. Question set: `part1-v2-20261113`.
These are measured development-set results on Windows RTX4090. Content/citation review is explicitly AI-assisted and provisional; it is not the required two-member human evaluation. The question draft needs team authorship, originality and reference review. The retained initial run and recorded fixes prevent a claim that this is an independent holdout.

| Measurement | Dense only | Final |
| --- | ---: | ---: |
| Correct / 25 (provisional AI content review) | 6 | 18 |
| Correct fraction | 0.2400 | 0.7200 |
| Recall@5 | 0.4722 | 0.7639 |
| Evidence-bearing retrieval cases | 24 | 24 |
| Citation accuracy (provisional AI source review) | 0.1406 | 0.4898 |
| Emitted citations reviewed | 64 | 49 |
| Median seconds / question | 6.1083 | 0.0393 |
| Two-member mean groundedness | Pending | Pending |
| Two-member agreement within one point | Pending | Pending |


The practice output has 40 rows in 35 conversations and passed the official provided format validator. Format validation does not establish answer correctness. The complete inference notebook has 46 executed code cells, with no error outputs. Its wall time was 428.27 seconds, including setup, practice and both evaluation modes.

Peak PyTorch allocated memory was 13.157 GiB and peak reserved memory was 13.836 GiB under a 14.0 GiB allocator budget. Text/visual retrieval ran on CPU and Qwen ran on CUDA device 0. This capacity-limited run does not verify actual T4/P100 hardware, runtime or Kaggle submission.

Across the 50 golden executions, emitted DOC3:p7 citations and Figure C-1 references were both 0; the maximum answer length was 97 whitespace-separated words. Appendix exclusion is prevention: its hidden image instruction was not supplied as trusted model input. No claim of robustness to arbitrary unseen image injections follows.

Recall@5 uses the frozen interleaved page/visual ranking before structured evidence expansion; its denominator excludes the no-evidence question. Citation judgments inspect material claim support, not mere source presence. Review notes and actual outputs are retained for team confirmation. Human metrics remain pending until two distinct team members submit all ratings.

All numerical values in this document are generated by this notebook from saved records. Do not type replacement results into the report. The team still needs to interpret observed failures in its own words, complete independent ratings, review the AI-assisted question set, and verify the tagged Kaggle run and final selected submission.


Measured artifacts: C:\Users\endle\Documents\Codex\2026-11-13\data266-lab2-part3-windows-ashley96yy-data266\outputs\part1\results_55942be


In [5]:
from src.p1_evaluation import numeric_targets_match
details=[]
for mode, run in runs.items():
    for record, case, review in zip(run['records'],spec['cases'],reviews[mode]['records']):
        answer=record['turn']['answer']
        generation=record['turn'].get('generation') or {}
        extraction=record['turn'].get('override_result') or {}
        called_vlm=bool(generation.get('input_tokens',0) or extraction.get('input_tokens',0))
        status_ok=answer['status']==case['status']
        numeric_ok=numeric_targets_match(answer['answer'],case.get('numeric_targets',[]))
        correct=review['answer_correct'] and status_ok and numeric_ok
        details.append(dict(mode=mode,question_id=case['question_id'],category=case['category'],
            correct=correct,status_correct=status_ok,numeric_targets_match=numeric_ok,
            called_vlm=called_vlm,elapsed_seconds=record['elapsed_seconds'],
            expected_status=case['status'],actual_status=answer['status'],
            question=case['question'],candidate=answer['answer'],reference=case['reference_answer'],
            ai_content_review_note=review['content_review_note']))
detail=pd.DataFrame(details)
detail.to_csv(OUTPUT/'per_question_review.csv',index=False)
category=detail.groupby(['category','mode'],sort=False)['correct'].agg(['sum','count']).reset_index()
category.to_csv(OUTPUT/'category_results.csv',index=False)
latency=detail.groupby(['mode','called_vlm'],sort=False)['elapsed_seconds'].agg(['count','median']).reset_index()
latency.to_csv(OUTPUT/'latency_breakdown.csv',index=False)
display(category)
display(latency)
print('The final overall median can be very low because many cases use deterministic evidence resolution. It is not a median VLM-call latency.')


,category,mode,sum,count
0,facts_tables,dense_only,4,5
1,figures,dense_only,0,6
2,two_sources,dense_only,0,3
3,multi_turn,dense_only,1,5
4,conflicts_revisions,dense_only,0,3
5,safety_absence,dense_only,1,3
6,facts_tables,final,5,5
7,figures,final,2,6
8,two_sources,final,2,3
9,multi_turn,final,5,5


,mode,called_vlm,count,median
0,dense_only,True,25,6.108288
1,final,False,14,0.032730
2,final,True,11,7.903862


The final overall median can be very low because many cases use deterministic evidence resolution. It is not a median VLM-call latency.


In [6]:
extra='\n## Timing interpretation and observed failure groups\n\n'
for row in latency.to_dict('records'):
    label='with VLM calls' if row['called_vlm'] else 'without VLM calls'
    extra+=f"- {row['mode']}: {row['count']} cases {label}; median {row['median']:.4f} seconds.\n"
extra+='\nThe overall final median includes direct evidence answers and session-state operations, which can avoid generation. Do not describe it as model-only inference speed.\n\n| Category | Mode | Correct | Cases |\n| --- | --- | ---: | ---: |\n'
for row in category.to_dict('records'):
    extra+=f"| {row['category']} | {row['mode']} | {row['sum']} | {row['count']} |\n"
extra+='\n## Actual final-system failures in this run\n\n'
failed=detail[(detail['mode']=='final') & ~detail['correct']]
for row in failed.to_dict('records'):
    extra+=f"- {row['question_id']}: {row['ai_content_review_note']}\n"
extra+='\nThese are provisional AI content judgments, not student groundedness ratings or a hidden-set grade. Keep the recorded failures visible; format validation alone is not a final-quality claim.\n'
target=OUTPUT/'PART1_MEASURED_RESULTS.md'
target.write_text(target.read_text(encoding='utf-8')+extra,encoding='utf-8')
display(Markdown(extra))



## Timing interpretation and observed failure groups

- dense_only: 25 cases with VLM calls; median 6.1083 seconds.
- final: 14 cases without VLM calls; median 0.0327 seconds.
- final: 11 cases with VLM calls; median 7.9039 seconds.

The overall final median includes direct evidence answers and session-state operations, which can avoid generation. Do not describe it as model-only inference speed.

| Category | Mode | Correct | Cases |
| --- | --- | ---: | ---: |
| facts_tables | dense_only | 4 | 5 |
| figures | dense_only | 0 | 6 |
| two_sources | dense_only | 0 | 3 |
| multi_turn | dense_only | 1 | 5 |
| conflicts_revisions | dense_only | 0 | 3 |
| safety_absence | dense_only | 1 | 3 |
| facts_tables | final | 5 | 5 |
| figures | final | 2 | 6 |
| two_sources | final | 2 | 3 |
| multi_turn | final | 5 | 5 |
| conflicts_revisions | final | 1 | 3 |
| safety_absence | final | 3 | 3 |

## Actual final-system failures in this run

- P1V2-06: Correct horizons but still falsely says both figures mark forecast years. A-3 is outturns. DOC2:p4 contradicts that central claim; extra citations do not support it.
- P1V2-09: Unwarranted abstention; both requested country changes are supported by the available bars.
- P1V2-10: Correct intercepts and direction, but omits the explicitly required 0.11-point change under the frozen numeric targets. Both cited pages support the facts stated; completeness and citation support are scored separately.
- P1V2-11: Correct stated GDP-deflator/headline readings, but still omits the B-4 core-CPI series and 1.8% value. Only DOC3:p4 supports these specific observations.
- P1V2-13: Wrong -1.7-point change for actual 2011-2013 unemployment and omits participation values, periods and explanation. No cited source supports that material computed outturn comparison.
- P1V2-20: Reports both source values but invents an explicit unemployment revision and uses answered instead of conflict. DOC1:p5/DOC3:p5 support their respective values; DOC2:p3 contradicts the fabricated revision claim.
- P1V2-22: Unwarranted abstention on the erratum. Corrected chart evidence was retrieved, but the output does not give the correction scope or exact label.

These are provisional AI content judgments, not student groundedness ratings or a hidden-set grade. Keep the recorded failures visible; format validation alone is not a final-quality claim.
